## Topic: Markdown Splitting

### 1. Introduction of Markdown Splitting

- Definition:
    - MarkdownHeaderTextSplitter splits Markdown (.md) files based on their header levels (#, ##, ###, ####, etc.). It also preserves the header hierarchy in each chunk's metadata.


- When to Use It?
    - Splitting README files, documentation, wikis, or Markdown notes.
    - When you want to preserve sections and subsections.
    - For technical documentation written in Markdown.


- How It Works
    - Scans for Markdown headers (#, ##, ###...).
    - Whenever it encounters a header, it starts a new chunk.
    - All content under that header belongs to that chunk.
    - Builds a header path and stores it in chunk.metadata.

### Example 1

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter,Language

text = """
# Project Name: Smart Student Tracker

A simple Python-based project to manage and track student data, including their grades, age, and academic status.


## Features

- Add new students with relevant info
- View student details
- Check if a student is passing
- Easily extendable class-based design


## 🛠 Tech Stack

- Python 3.10+
- No external dependencies


## Getting Started

1. Clone the repo  
   ```bash
   git clone https://github.com/your-username/student-tracker.git

"""

# Initialize the splitter
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=200,
    chunk_overlap=0,
)

# Perform the split
chunks = splitter.split_text(text)

print(len(chunks))
print(chunks[0])

3
# Project Name: Smart Student Tracker

A simple Python-based project to manage and track student data, including their grades, age, and academic status.


In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_text = """
# Machine Learning Guide

## Introduction
Machine learning is a subset of Artificial Intelligence (AI). It enables 
computers to learn patterns from data without being explicitly programmed.

## Types of Machine Learning

### Supervised Learning
Supervised learning uses labeled datasets. Common examples are 
Classification and Regression problems.

### Unsupervised Learning
Unsupervised learning works with unlabeled data. It is used for 
Clustering, Dimensionality Reduction, and Anomaly Detection.

## Getting Started
To get started with Machine Learning, learn Python, NumPy, Pandas, 
and Scikit-Learn.
"""

# Define header levels to split on
headers_to_split_on = [
    ("#", "Header 1"),      # H1
    ("##", "Header 2"),     # H2
    ("###", "Header 3"),    # H3
]

# Create the splitter
splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=True
)

# Split the Markdown text
chunks = splitter.split_text(markdown_text)

print(f"Total Number of Chunk: {len(chunks)}")

# View the chunks
for i, chunk in enumerate(chunks):
    print("=" * 60)
    print(f"CHUNK {i+1}")
    print("=" * 60)
    print("METADATA:", chunk.metadata)
    print("CONTENT:")
    print(chunk.page_content)
    print()

Total Number of Chunk: 4
CHUNK 1
METADATA: {'Header 1': 'Machine Learning Guide', 'Header 2': 'Introduction'}
CONTENT:
Machine learning is a subset of Artificial Intelligence (AI). It enables
computers to learn patterns from data without being explicitly programmed.

CHUNK 2
METADATA: {'Header 1': 'Machine Learning Guide', 'Header 2': 'Types of Machine Learning', 'Header 3': 'Supervised Learning'}
CONTENT:
Supervised learning uses labeled datasets. Common examples are
Classification and Regression problems.

CHUNK 3
METADATA: {'Header 1': 'Machine Learning Guide', 'Header 2': 'Types of Machine Learning', 'Header 3': 'Unsupervised Learning'}
CONTENT:
Unsupervised learning works with unlabeled data. It is used for
Clustering, Dimensionality Reduction, and Anomaly Detection.

CHUNK 4
METADATA: {'Header 1': 'Machine Learning Guide', 'Header 2': 'Getting Started'}
CONTENT:
To get started with Machine Learning, learn Python, NumPy, Pandas,
and Scikit-Learn.



In [ ]:
# Keeping Headers in Content 

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2")],
    strip_headers=False  # Keep # and ## in the content
)

chunks = splitter.split_text(markdown_text)
print(chunks[0].page_content)

# Machine Learning Guide  
## Introduction
Machine learning is a subset of Artificial Intelligence (AI). It enables
computers to learn patterns from data without being explicitly programmed.


- Key Takeaway
    - MarkdownHeaderTextSplitter does NOT use chunk_size or chunk_overlap. It splits ONLY by Markdown headers. If a single section is extremely long, you'll need to combine it with RecursiveCharacterTextSplitter later.

### Example 2: Combining with RecursiveCharacterTextSplitte

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter

markdown_text = """
# Machine Learning Guide

## Introduction
Machine learning is a subset of Artificial Intelligence (AI). It enables 
computers to learn patterns from data without being explicitly programmed.

## Types of Machine Learning

### Supervised Learning
Supervised learning uses labeled datasets. Common examples are 
Classification and Regression problems.

### Unsupervised Learning
Unsupervised learning works with unlabeled data. It is used for 
Clustering, Dimensionality Reduction, and Anomaly Detection.

## Getting Started
To get started with Machine Learning, learn Python, NumPy, Pandas, 
and Scikit-Learn.
"""

# Step 1: Split by Markdown headers
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2")]
)
header_chunks = header_splitter.split_text(markdown_text)

# Step 2: If any chunk is too large, split it further
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

final_chunks = []
for chunk in header_chunks:
    # Split this large section chunk
    sub_chunks = text_splitter.split_text(chunk.page_content)
    # Preserve the original metadata
    for sc in sub_chunks:
        final_chunks.append(type(chunk)(page_content=sc, metadata=chunk.metadata.copy()))

print(f"Final chunks: {len(final_chunks)}")

print(final_chunks)

Final chunks: 7
[Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Introduction'}, page_content='Machine learning is a subset of Artificial Intelligence (AI). It enables'), Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Introduction'}, page_content='computers to learn patterns from data without being explicitly programmed.'), Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Types of Machine Learning'}, page_content='### Supervised Learning\nSupervised learning uses labeled datasets. Common examples are'), Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Types of Machine Learning'}, page_content='Classification and Regression problems.  \n### Unsupervised Learning'), Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Types of Machine Learning'}, page_content='Unsupervised learning works with unlabeled data. It is used for'), Document(metadata={'H1': 'Machine Learning Guide', 'H2': 'Types of Machine Learning'}, page_content='Clustering, Dimensi